# 전체 입력 변수 사용 적절성 평가
112개 원자료의 정의·지역·품질·가용성, 78개 조건부 후보의 예측 기여도를 확인합니다.
본 노트북은 완료된 실험을 읽으며 모델 학습이나 제출 파일 수정은 하지 않습니다. 전체 재현: 프로젝트 루트에서 `python -m src.variable_suitability`.
**주의:** 34개 보류는 예측력이 없다는 의미가 아닙니다. 모든 입력의 정확한 공표 이력·개정 빈티지는 미확인입니다. 탐색 결과를 최종 변수 선택 성능으로 해석하지 않습니다.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
OUT = ROOT / 'outputs_variable_suitability'
audit = pd.read_csv(OUT / 'variable_assessment.csv')
metrics = pd.read_csv(OUT / 'metrics.csv')
comparisons = pd.read_csv(OUT / 'comparisons.csv')
redundancy = pd.read_csv(OUT / 'redundancy.csv')
manifest = json.loads((OUT / 'manifest.json').read_text(encoding='utf-8'))
assert len(audit) == 112 and audit.variable.is_unique
display(audit.groupby(['group', 'gate']).size().unstack(fill_value=0))
print('전체 입력:', len(audit), '조건부 실험:', audit.predictive_eligible.sum(), '보류:', (~audit.predictive_eligible).sum())

gate,보류,조건부 실험
group,,
activity_unknown,4,0
cli_foreign,0,41
cli_us,0,8
commodities,0,4
energy,1,3
eu_storage,2,0
financial,0,8
fundamentals,0,6
macro,0,6


전체 입력: 112 조건부 실험: 78 보류: 34


## 1. 정의·가용성·품질 게이트
실험 진입은 초기 120개월 관측률 80% 이상·비상수 관측·대상 국가 명시 조건으로 결정했습니다. 전체 훈련자료 결측률은 설명용입니다. 아래 보류 변수는 자동 보간하여 모델에 넣지 않았습니다.

In [2]:
display(audit.loc[~audit.predictive_eligible, ['variable', 'unit', 'region', 'initial120_coverage_pct', 'train204_missing_pct', 'gate_reason']])
important = ['high_temperature', 'low_temperature', 'Electricity_Price_USA', 'Total_Natural_Gas_Underground_Storage_Volume_USA', 'Total_Natural_Gas_Marketed_Production_USA', 'USDEUR_FOREX']
with pd.option_context('display.max_colwidth', 180):
    display(audit.loc[audit.variable.isin(important), ['variable', 'unit', 'description', 'lag_months_assumed', 'caveats']])

,variable,unit,region,initial120_coverage_pct,train204_missing_pct,gate_reason
3,Heating_Oil_Price_USA,Dollars per Gallon($/GAL),USA,50.000000,50.000000,초기120개월 관측률80% 미만
77,Comprehensive_PMI,%,미확인,0.000000,78.431373,국가/대상 경제 미확인; 초기120개월 관측률80% 미만; 초기120개월 비상수 관...
78,Construction_PMI,%,미확인,0.000000,100.000000,국가/대상 경제 미확인; 초기120개월 관측률80% 미만; 초기120개월 비상수 관...
79,M_PMI_Comp_Employment,%,미확인,86.666667,7.843137,국가/대상 경제 미확인
80,M_PMI_Comp_Expected_Production_and_Business_Ac...,%,미확인,6.666667,54.901961,국가/대상 경제 미확인; 초기120개월 관측률80% 미만
81,M_PMI_Comp_Finished_Goods_Inventory,%,미확인,86.666667,7.843137,국가/대상 경제 미확인
82,M_PMI_Comp_Import,%,미확인,86.666667,7.843137,국가/대상 경제 미확인
83,M_PMI_Comp_In_Hand_Orders,%,미확인,86.666667,7.843137,국가/대상 경제 미확인
84,M_PMI_Comp_Main_Raw_Material_Purchase_Price,%,미확인,86.666667,7.843137,국가/대상 경제 미확인
85,M_PMI_Comp_New_Export_Orders,%,미확인,86.666667,7.843137,국가/대상 경제 미확인


,variable,unit,description,lag_months_assumed,caveats
2,Electricity_Price_USA,cents per kilowatthour,Electricity Sales To Ultimate Customers Average,2,단위 cents/kWh는 가격; 한국어 판매량 설명과 불일치
8,Total_Natural_Gas_Marketed_Production_USA,Million Cubic Feat(MMCF),U.S. Natural Gas Marketed Production,2,marketed production은 판매량/건성가스 생산량과 구분
9,Total_Natural_Gas_Underground_Storage_Volume_USA,Million Cubic Feat(MMCF),U.S. Natural Gas Underground Storage Volume (MMcf),2,stock; 총가스/working gas 구분 및 월말값 여부 미확인; 기존 lag1 재검증
20,USDEUR_FOREX,Index,USD to EUROPEAN UNION Euro,0,단위 Index와 환율 설명 불일치; 호가 방향 확인 필요
110,high_temperature,℉,"The highest monthly temperature in Boston, USA",1,보스턴 월 극값; 전국 평균 기온/실제 HDD·CDD 아님
111,low_temperature,℉,"The lowest monthly temperature in Boston, USA",1,보스턴 월 극값; 전국 평균 기온/실제 HDD·CDD 아님


## 2. 중복성과 공통 조건 예측 비교
상관은 초기 120개월에서 계산했으며 수준과 차분을 구분했습니다. 수준 상관이 높다는 이유만으로 제거하지 않았습니다. 예측 비교는 Ridge/LightGBM에 같은 특징을 넣어 H1/H2/H3를 따로 계산했습니다. `add`는 가격 이력에 추가, `drop`은 전체 적격 변수 조합에서 제거, `delay`는 해당 변수군을 한 달 더 지연하여 가격 이력에 추가한 결과입니다.

In [3]:
display(redundancy)
overall = metrics.loc[metrics.period == 'all']
group_names = ['naive', 'history', 'all_groups', 'processed_reference', 'processed_storage_lag2']
group_results = overall.loc[overall.candidate.isin(group_names) | overall.candidate.str.startswith('add_')]
display(group_results.pivot(index=['family', 'candidate'], columns='horizon', values='mape_pct').round(3))
display(comparisons.loc[comparisons.candidate == 'processed_storage_lag2'].round(3))

,kind,variable_a,variable_b,correlation,paired_n,exact_duplicate
0,level,Brent_Crude_Oil,WTI_Crude_Oil,0.967997,120,False
1,level,Brent_Crude_Oil,Heating_Oil_Price_USA,0.956220,60,False
2,level,Heating_Oil_Price_USA,CPI_Energy_Seasonally_Adjusted_USA,0.980945,60,False
3,level,Heating_Oil_Price_USA,PPI_Manufacturing_Sector_USA,0.966663,60,False
4,level,BCOMGC_INDX,CLI_Comp_Construction_Production_CHN,0.951447,120,False
5,level,DJI_INDX,GSPC_INDX,0.975977,120,False
6,level,CPI_Index_Seasonally_Adjusted_USA,PPI_Manufacturing_Sector_USA,0.985721,120,False
7,level,CLI_Comp_Business_Situation_DEU,CLI_Comp_Finished_Goods_Stocks_Tendency_DEU,-0.980582,120,False
8,level,CLI_Comp_Business_Situation_DEU,Composite_Leading_Indicator_DEU,0.960134,120,False
9,level,CLI_Comp_Export_Orders_Tendency_DEU,CLI_Comp_Stocks_JPN,-0.956196,120,False


horizon                               1       2       3
family   candidate                                     
lightgbm add_cli_foreign          7.703  11.553  14.716
         add_cli_us               7.756  10.958  14.952
         add_commodities          7.297  10.515  14.420
         add_energy               7.449  10.633  13.820
         add_financial            7.822  10.830  15.253
         add_fundamentals         7.449  10.943  14.801
         add_macro                7.970  11.556  15.150
         add_weather              7.568  11.207  15.585
         all_groups               7.057  10.514  14.552
         history                  7.678  11.151  15.763
         naive                    7.666  10.480  13.570
         processed_reference      7.043   9.808  13.668
         processed_storage_lag2   7.087   9.781  13.540
ridge    add_cli_foreign         10.029  19.920  82.764
         add_cli_us              11.360  24.701  17.178
         add_commodities          7.699  10.482  13.541
         add_energy               7.668   9.488  11.837
         add_financial            8.403  11.305  15.187
         add_fundamentals         7.292   9.337  11.484
         add_macro                7.726  10.695  13.784
         add_weather              7.838  11.092  14.693
         all_groups              10.017  26.632  52.519
         history                  7.882  11.036  14.634
         naive                    7.666  10.480  13.570
         processed_reference      6.752   9.202  11.905
         processed_storage_lag2   6.770   9.212  11.723

,family,candidate,horizon,reference,delta_mape_pp,early_delta_pp,late_delta_pp,years_better,n_years,block6_low_pp,block6_high_pp
78,lightgbm,processed_storage_lag2,1,processed_reference,0.044,-0.013,0.100,4,8,-0.118,0.276
79,lightgbm,processed_storage_lag2,2,processed_reference,-0.028,-0.175,0.120,3,8,-0.294,0.414
80,lightgbm,processed_storage_lag2,3,processed_reference,-0.128,-0.364,0.114,3,8,-1.024,1.057
393,ridge,processed_storage_lag2,1,processed_reference,0.018,-0.056,0.091,4,8,-0.110,0.139
394,ridge,processed_storage_lag2,2,processed_reference,0.010,0.049,-0.029,4,8,-0.123,0.125
395,ridge,processed_storage_lag2,3,processed_reference,-0.182,-0.183,-0.182,5,8,-0.374,-0.038


## 3. 변수별 결과와 안정성
ΔMAPE는 후보−동일 모델의 가격 이력 기준선으로 음수이면 개선입니다. 6개 조합은 2개 모델×3개 horizon입니다. 안정성 표시는 전체·전후반 개선과 4개 이상 연도 개선을 동시에 만족하는 진단입니다. 확정 채택 또는 다중 비교 보정된 유의성 기준이 아닙니다. 원시 수준+차분을 단독 추가한 결과와 기존 가공 변수 조합에서의 제거 실험은 서로 다른 질문입니다.

In [4]:
display(audit[['variable', 'group', 'gate', 'single_improving_of6', 'single_stable_of6', 'predictive_evidence', 'decision']])
# 아래 변수명을 바꾸면 6개 모델/horizon의 오차·전후반·연도·bootstrap 진단을 볼 수 있습니다.
variable = 'Total_Natural_Gas_Underground_Storage_Volume_USA'
variable_id = audit.set_index('variable').loc[variable, 'variable_id']
display(comparisons.loc[comparisons.candidate == 'single_' + variable_id].round(4))
print('전체 표/설명:', OUT / 'RESULTS.md')
print('평가 결과를 바탕으로 최종 모델이나 submission.csv를 자동 변경하지 않았습니다.')

,variable,group,gate,single_improving_of6,single_stable_of6,predictive_evidence,decision
0,Brent_Crude_Oil,energy,조건부 실험,3.0,3.0,단독추가 개선 3/6; 전후반·연도 안정 3/6,조건부 재검증 후보
1,WTI_Crude_Oil,energy,조건부 실험,3.0,2.0,단독추가 개선 3/6; 전후반·연도 안정 2/6,조건부 재검증 후보
2,Electricity_Price_USA,energy,조건부 실험,5.0,5.0,단독추가 개선 5/6; 전후반·연도 안정 5/6,조건부 재검증 후보
3,Heating_Oil_Price_USA,energy,보류,NaN,NaN,미실험(게이트 보류); 무용함을 의미하지 않음,보류: 초기120개월 관측률80% 미만
4,Natural_Gas_Imports_From_Canada_USA,fundamentals,조건부 실험,6.0,4.0,단독추가 개선 6/6; 전후반·연도 안정 4/6,조건부 재검증 후보
...,...,...,...,...,...,...,...
107,Motor_Vehicles_Accumulated_YoY,activity_unknown,보류,NaN,NaN,미실험(게이트 보류); 무용함을 의미하지 않음,보류: 국가/대상 경제 미확인
108,EU_Aggregated_Gas_Storage_Inventory,eu_storage,보류,NaN,NaN,미실험(게이트 보류); 무용함을 의미하지 않음,보류: 초기120개월 관측률80% 미만
109,EU_Aggregated_LNG_System_Inventory,eu_storage,보류,NaN,NaN,미실험(게이트 보류); 무용함을 의미하지 않음,보류: 초기120개월 관측률80% 미만
110,high_temperature,weather,조건부 실험,3.0,0.0,단독추가 개선 3/6; 전후반·연도 안정 0/6,현 표현에서 안정 개선 근거 부족


,family,candidate,horizon,reference,delta_mape_pp,early_delta_pp,late_delta_pp,years_better,n_years,block6_low_pp,block6_high_pp
105,lightgbm,single_v009,1,history,0.0343,-0.1168,0.1818,4,8,-0.2554,0.3061
106,lightgbm,single_v009,2,history,-0.1914,-0.3116,-0.0713,5,8,-0.6342,0.2879
107,lightgbm,single_v009,3,history,0.3287,0.1056,0.5573,2,8,-0.2533,0.7200
420,ridge,single_v009,1,history,0.0294,0.2087,-0.1457,4,8,-0.4453,0.4499
421,ridge,single_v009,2,history,-0.3039,-0.0683,-0.5396,6,8,-1.3655,0.7961
422,ridge,single_v009,3,history,-0.7831,-0.6847,-0.8839,5,8,-1.9336,0.3014


전체 표/설명: C:\Users\Laptop PC\Desktop\nature_gas_prediction\outputs_variable_suitability\RESULTS.md
평가 결과를 바탕으로 최종 모델이나 submission.csv를 자동 변경하지 않았습니다.
